# BUSINESS UNDERSTANDING

#### > Food delivery adalah layanan yang menghubungkan customer dengan merchant/restoran melalui platform online.

#### > Platform seperti Zomato menghasilkan data transaksi yang bisa dianalisis untuk meningkatkan performa bisnis.

#### > Project ini bertujuan menganalisis order food delivery untuk menemukan insight bagi tim product, merchant, operations, dan marketing.

Dataset Kaggle: https://www.kaggle.com/datasets/sujalsuthar/food-delivery-order-history-data

### Business Problem

Bagaimana platform food delivery dapat menggunakan data transaksi untuk meningkatkan customer engagement, merchant performance, campaign timing, dan delivery experience?

### Pertanyaan Bisnis

1. Bagaimana distribusi order value?
2. Area (subzone) mana yang menghasilkan order dan revenue tertinggi?
3. Restoran mana yang memiliki performa terbaik?
4. Kapan waktu puncak order berdasarkan hari dan jam?
5. Apakah promo/discount berkaitan dengan order value?
6. Faktor apa yang paling berpengaruh terhadap high value order?

# Load Dataset

In [ ]:
import os

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

In [ ]:
import pandas as pd
import numpy as np

csv_files = []

for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        if filename.lower().endswith('.csv'):
            csv_files.append(os.path.join(dirname, filename))

print("CSV files ditemukan:")
for path in csv_files:
    print(path)

if len(csv_files) == 0:
    raise FileNotFoundError("Tidak ada file CSV di /kaggle/input.")

data_path = csv_files[0]
df = pd.read_csv(data_path)
df.head(10)

# EDA (Exploratory Data Analysis)

In [ ]:
# ukuran data (baris, kolom)
df.shape

In [ ]:
# melihat jumlah baris, kolom, dan tipe data
df.info()

### Penjelasan Kolom

> `Bill subtotal` adalah total harga item sebelum diskon (order value).

> `Total` adalah harga akhir yang dibayar customer setelah diskon dan packaging.

> `Restaurant discount (Promo)` adalah diskon dari restoran berupa promo.

> `KPT duration (minutes)` adalah Kitchen Preparation Time, waktu restoran menyiapkan pesanan.

> `Rider wait time (minutes)` adalah waktu rider menunggu di restoran.

> `Rating` adalah rating yang diberikan customer (1-5).

> `Distance` adalah jarak pengiriman (dalam km).

> `Order Ready Marked` menunjukkan apakah restoran menandai order ready dengan benar atau tidak.

## DATA UNDERSTANDING

#### 1. Apakah ada missing value?
#### 2. Apakah ada duplikasi?
#### 3. Bagaimana distribusi order value?
#### 4. Area mana yang paling banyak order?
#### 5. Restoran mana yang performanya terbaik?

In [ ]:
# ringkasan deskriptif numerik
df.describe()

In [ ]:
# ringkasan kolom kategorikal
df.describe(include='object')

In [ ]:
# Mencari nilai kosong
print("Missing Values:")
print(df.isna().sum())

In [ ]:
# Mencari duplikasi baris penuh
print("Duplicate Values:")
print(df.duplicated().sum())

In [ ]:
# jumlah nilai unik di setiap kolom
df.nunique()

## Data Cleaning

> Dataset ini memiliki banyak missing value pada kolom opsional seperti `Rating`, `Review`, `Instructions`, dan `Customer complaint tag`. Ini wajar karena tidak semua customer memberikan rating atau review.

> Kolom utama seperti `Bill subtotal`, `Total`, `Restaurant name`, `Subzone`, dan `Order Placed At` tidak memiliki missing value.

> Tidak ada duplikasi baris.

> Nama kolom dibersihkan agar mudah dipanggil (lowercase, snake_case).

In [ ]:
df_clean = df.copy()

# Bersihkan nama kolom
df_clean.columns = (
    df_clean.columns
    .str.strip()
    .str.lower()
    .str.replace(' ', '_', regex=False)
    .str.replace('(', '', regex=False)
    .str.replace(')', '', regex=False)
    .str.replace('/', '_', regex=False)
    .str.replace('&', 'and', regex=False)
)

print("Data awal:", df_clean.shape)
print("\nNama kolom setelah cleaning:")
print(df_clean.columns.tolist())

In [ ]:
# Konversi kolom datetime
# Format asli: "11:38 PM, September 10 2024"
df_clean['order_placed_at'] = pd.to_datetime(
    df_clean['order_placed_at'],
    format='%I:%M %p, %B %d %Y',
    errors='coerce'
)

print("Kolom order_placed_at berhasil dikonversi:", df_clean['order_placed_at'].notna().sum(), "dari", len(df_clean))

In [ ]:
# Bersihkan kolom distance: "3km" -> 3.0, "<1km" -> 0.5
df_clean['distance_km'] = (
    df_clean['distance']
    .str.replace('km', '', regex=False)
    .str.replace('<1', '0.5', regex=False)
    .str.strip()
)
df_clean['distance_km'] = pd.to_numeric(df_clean['distance_km'], errors='coerce')

print("Contoh distance setelah cleaning:")
print(df_clean[['distance', 'distance_km']].drop_duplicates().head(10))

In [ ]:
# Buat total_discount = semua jenis diskon dijumlahkan
discount_cols = [col for col in df_clean.columns if 'discount' in col and col != 'discount_construct']
print("Kolom diskon yang ditemukan:", discount_cols)

df_clean['total_discount'] = df_clean[discount_cols].sum(axis=1)

print("\nMissing Values kolom utama:")
print(df_clean[['bill_subtotal', 'total', 'total_discount', 'rating', 'kpt_duration_minutes', 'distance_km']].isna().sum())

## Feature Engineering

> `order_hour` dan `order_day` diekstrak dari kolom waktu order untuk analisis waktu puncak.

> `is_weekend` menandai apakah order dilakukan di akhir pekan.

> `has_discount` menandai apakah order menggunakan diskon (total_discount > 0).

> `high_value_order` adalah target klasifikasi. Order dianggap high value jika `bill_subtotal >= median`.

In [ ]:
# Ekstrak fitur waktu
df_clean['order_hour'] = df_clean['order_placed_at'].dt.hour
df_clean['order_day'] = df_clean['order_placed_at'].dt.day_name()
df_clean['is_weekend'] = df_clean['order_day'].isin(['Saturday', 'Sunday']).astype(int)

# Has discount
df_clean['has_discount'] = (df_clean['total_discount'] > 0).astype(int)

# Target: high value order (di atas median bill_subtotal)
median_value = df_clean['bill_subtotal'].median()
df_clean['high_value_order'] = (df_clean['bill_subtotal'] >= median_value).astype(int)

print(f"Median order value: {median_value}")

df_clean[['bill_subtotal', 'total_discount', 'has_discount', 'order_hour', 'order_day', 'is_weekend', 'high_value_order']].head()

# Target Variable: High Value Order

In [ ]:
df_clean['high_value_order'].value_counts()

In [ ]:
df_clean['high_value_order'].value_counts(normalize=True) * 100

> Target `high_value_order` digunakan untuk melihat pola order bernilai tinggi.

> Order dengan `bill_subtotal` di atas median dianggap sebagai high value order.

> Ini relevan untuk bisnis karena order value tinggi berkaitan langsung dengan revenue dan merchant performance.

## Visualisasi Order Value

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 5))
sns.histplot(data=df_clean, x='bill_subtotal', bins=30, kde=True)
plt.title('Distribusi Order Value (Bill Subtotal)')
plt.xlabel('Bill Subtotal')
plt.ylabel('Jumlah Order')
plt.show()

In [ ]:
sns.countplot(data=df_clean, x='high_value_order')
plt.title('Distribusi High Value Order')
plt.xlabel('High Value Order (0 = Low, 1 = High)')
plt.ylabel('Jumlah Order')
plt.show()

## Analisis Area (Subzone)

> Kolom `city` hanya berisi 1 nilai (Delhi NCR), sehingga tidak informatif untuk perbandingan.

> Sebagai gantinya, kita gunakan `subzone` yang memiliki 8 area berbeda.

In [ ]:
area_summary = (
    df_clean.groupby('subzone')['bill_subtotal']
    .agg(['count', 'sum', 'mean'])
    .sort_values('sum', ascending=False)
)
area_summary.columns = ['total_orders', 'total_revenue', 'avg_order_value']
area_summary

In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(data=area_summary.reset_index(), y='subzone', x='total_revenue')
plt.title('Revenue Berdasarkan Subzone')
plt.xlabel('Total Revenue')
plt.ylabel('Subzone')
plt.show()

## Analisis Restoran

In [ ]:
restaurant_summary = (
    df_clean.groupby('restaurant_name')
    .agg(
        total_orders=('bill_subtotal', 'count'),
        total_revenue=('bill_subtotal', 'sum'),
        avg_order_value=('bill_subtotal', 'mean'),
        avg_rating=('rating', 'mean')
    )
    .sort_values('total_revenue', ascending=False)
)
restaurant_summary

In [ ]:
plt.figure(figsize=(10, 6))
sns.barplot(data=restaurant_summary.reset_index(), y='restaurant_name', x='total_revenue')
plt.title('Revenue Berdasarkan Restoran')
plt.xlabel('Total Revenue')
plt.ylabel('Restoran')
plt.show()

## Analisis Waktu Puncak Order

In [ ]:
hourly_orders = (
    df_clean.groupby('order_hour')['bill_subtotal']
    .agg(['count', 'sum'])
    .reset_index()
)
hourly_orders.columns = ['order_hour', 'total_orders', 'total_revenue']

plt.figure(figsize=(10, 5))
sns.lineplot(data=hourly_orders, x='order_hour', y='total_orders', marker='o')
plt.title('Jumlah Order Berdasarkan Jam')
plt.xlabel('Jam Order')
plt.ylabel('Total Order')
plt.xticks(range(0, 24))
plt.show()

In [ ]:
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
daily_orders = (
    df_clean.groupby('order_day')['bill_subtotal']
    .agg(['count', 'sum'])
    .reindex(day_order)
    .reset_index()
)
daily_orders.columns = ['order_day', 'total_orders', 'total_revenue']

plt.figure(figsize=(10, 5))
sns.barplot(data=daily_orders, x='order_day', y='total_orders')
plt.title('Jumlah Order Berdasarkan Hari')
plt.xlabel('Hari')
plt.ylabel('Total Order')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()

## Analisis Promo / Discount

In [ ]:
promo_summary = (
    df_clean.groupby('has_discount')['bill_subtotal']
    .agg(['count', 'mean', 'median', 'sum'])
)
promo_summary

In [ ]:
plt.figure(figsize=(6, 4))
sns.boxplot(data=df_clean, x='has_discount', y='bill_subtotal')
plt.title('Order Value: Dengan Diskon vs Tanpa Diskon')
plt.xlabel('Has Discount (0 = No, 1 = Yes)')
plt.ylabel('Bill Subtotal')
plt.show()

## Mencari Kolom yang Berkaitan dengan Order Value Tinggi

> Heatmap korelasi digunakan untuk melihat hubungan antar fitur numerik dengan order value.

> Boxplot digunakan untuk memvisualisasikan perbedaan fitur antara high value order dan low value order.

In [ ]:
# Heatmap korelasi
corr_cols = ['bill_subtotal', 'total', 'total_discount', 'kpt_duration_minutes',
             'rider_wait_time_minutes', 'distance_km', 'order_hour', 'is_weekend', 'high_value_order']

plt.figure(figsize=(10, 8))
sns.heatmap(
    df_clean[corr_cols].corr(),
    annot=True,
    cmap='coolwarm',
    fmt='.2f'
)
plt.title('Correlation Heatmap')
plt.tight_layout()
plt.show()

In [ ]:
# Boxplot perbandingan fitur vs high_value_order
compare_cols = ['total_discount', 'kpt_duration_minutes', 'rider_wait_time_minutes',
                'distance_km', 'order_hour', 'is_weekend']

fig, axes = plt.subplots(nrows=2, ncols=3, figsize=(18, 8))
axes = axes.flatten()

for i, col in enumerate(compare_cols):
    sns.boxplot(data=df_clean, x='high_value_order', y=col, ax=axes[i])
    axes[i].set_title(f'{col} vs High Value Order')

plt.tight_layout()
plt.show()

## Persiapan Data untuk Modeling

> Target modeling adalah `high_value_order`.

> Kolom teks seperti `restaurant_name`, `order_id`, `customer_id`, `items_in_order` **tidak digunakan** karena:
> 1. **Nilai unik terlalu banyak** — menyebabkan overfitting.
> 2. **Tujuan analisis adalah mencari pola dari fitur operasional** (waktu, diskon, jarak, preparation time), bukan dari nama restoran tertentu.

> Kolom `subzone` di-encode menggunakan one-hot encoding karena jumlah area hanya 8 (masih wajar).

In [ ]:
model_features = [
    'total_discount', 'has_discount', 'kpt_duration_minutes',
    'rider_wait_time_minutes', 'distance_km', 'order_hour',
    'is_weekend', 'rating', 'subzone'
]

model_df = df_clean[model_features + ['high_value_order']].copy()
model_df = model_df.dropna()
model_df = pd.get_dummies(model_df, columns=['subzone'], drop_first=True)

print("Data untuk modeling:", model_df.shape)
model_df.head()

### Encode Target

In [ ]:
X = model_df.drop(columns=['high_value_order'])
y = model_df['high_value_order']

feature_names = X.columns.tolist()
print("Fitur yang digunakan:", feature_names)

## Pemisahan Data untuk Training

In [ ]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

## Logistic Regression (Baseline)

> Karena targetnya adalah high value order atau bukan, maka problem-nya adalah classification.

> Logistic Regression dipakai sebagai baseline karena mudah dijelaskan.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model_lr = LogisticRegression(max_iter=1000, class_weight='balanced', random_state=42)
model_lr.fit(X_train_scaled, y_train)

y_pred_lr = model_lr.predict(X_test_scaled)
print(classification_report(y_test, y_pred_lr))

## Model RandomForest

In [ ]:
from sklearn.ensemble import RandomForestClassifier

model_rf = RandomForestClassifier(
    n_estimators=200,
    class_weight='balanced',
    random_state=42,
    n_jobs=-1
)

model_rf.fit(X_train, y_train)

y_pred_rf = model_rf.predict(X_test)
print(classification_report(y_test, y_pred_rf))

## Model XGBoost

In [ ]:
try:
    from xgboost import XGBClassifier

    scale_pos_weight = (y_train == 0).sum() / (y_train == 1).sum()

    model_xgb = XGBClassifier(
        scale_pos_weight=scale_pos_weight,
        random_state=42,
        eval_metric='logloss'
    )

    model_xgb.fit(X_train, y_train)

    y_pred_xgb = model_xgb.predict(X_test)
    print(classification_report(y_test, y_pred_xgb))

except ImportError:
    print("XGBoost belum tersedia. Lewati model XGB.")

## Feature Importance


### Logistic Regression

In [ ]:
coef = pd.Series(np.abs(model_lr.coef_[0]), index=feature_names)

coef.nlargest(10).sort_values().plot(kind='barh', color='steelblue')
plt.title('Top 10 Fitur Paling Berpengaruh (Logistic Regression)')
plt.xlabel('Absolute Coefficient')
plt.tight_layout()
plt.show()

### RandomForest

In [ ]:
feat_importance_rf = pd.Series(model_rf.feature_importances_, index=feature_names)

feat_importance_rf.nlargest(10).sort_values().plot(kind='barh')
plt.title('Top 10 Fitur Paling Berpengaruh (RandomForest)')
plt.xlabel('Feature Importance')
plt.tight_layout()
plt.show()

### XGBoost

In [ ]:
if 'model_xgb' in globals():
    feat_importance_xgb = pd.Series(model_xgb.feature_importances_, index=feature_names)

    feat_importance_xgb.nlargest(10).sort_values().plot(kind='barh')
    plt.title('Top 10 Fitur Paling Berpengaruh (XGBoost)')
    plt.xlabel('Feature Importance')
    plt.tight_layout()
    plt.show()
else:
    print("Model XGB tidak tersedia.")

# Kesimpulan

In [ ]:
total_orders = len(df_clean)
total_revenue = df_clean['bill_subtotal'].sum()
avg_order_value = df_clean['bill_subtotal'].mean()
high_value_rate = df_clean['high_value_order'].mean() * 100

print("RINGKASAN BISNIS")
print(f"Total order setelah cleaning: {total_orders:,}")
print(f"Total revenue (bill subtotal): {total_revenue:,.2f}")
print(f"Average order value: {avg_order_value:,.2f}")
print(f"Median order value: {df_clean['bill_subtotal'].median():,.2f}")
print(f"Persentase high value order: {high_value_rate:.2f}%")

print("\nTop restoran berdasarkan revenue:")
print(restaurant_summary[['total_orders', 'total_revenue', 'avg_order_value']].head(5))

print("\nTop subzone berdasarkan revenue:")
print(area_summary.head(5))

print("\nRata-rata order value: diskon vs tanpa diskon:")
print(promo_summary[['mean', 'count']])

## Jawaban Bisnis

> **Catatan:** Isi jawaban di bawah ini harus disesuaikan dengan angka hasil output di atas setelah notebook dijalankan.

1. **Bagaimana distribusi order value?**
   Distribusi order value dilihat dari histogram dan median. Order di atas median dikategorikan sebagai high value order.

2. **Area mana yang menghasilkan revenue tertinggi?**
   Subzone dengan total revenue dan jumlah order tertinggi dapat diprioritaskan untuk ekspansi merchant dan campaign lokal.

3. **Restoran mana yang performanya terbaik?**
   Restoran dengan revenue tertinggi, jumlah order terbanyak, dan rating terbaik dapat dijadikan mitra prioritas untuk program visibility dan partnership.

4. **Kapan waktu puncak order?**
   Jam dan hari dengan jumlah order tertinggi digunakan untuk menentukan waktu optimal campaign, promo push notification, dan alokasi rider.

5. **Apakah promo efektif?**
   Perbandingan rata-rata order value antara order dengan diskon vs tanpa diskon menunjukkan apakah promo mendorong order bernilai lebih tinggi.

6. **Faktor apa yang paling berpengaruh terhadap high value order?**
   Feature importance dari model menunjukkan fitur operasional mana yang paling berkaitan dengan order bernilai tinggi (misalnya total_discount, distance, kpt_duration, dll).